# Multiple polynomiale Regression – Hypothesenräume vergleichen

In dieser Übung können Sie **eine oder mehrere numerische Eingabevariablen** auswählen. Für jeden Polynomgrad werden auch **Interaktionen zwischen den Features** erzeugt.

**Ziele:** Features auswählen, Modellkomplexität (Polynomgrad) ändern, RMSE auf Trainings- und Validierungsdaten vergleichen sowie Underfitting/Overfitting erkennen. Ein **unberührter Testdatensatz** dient zur abschließenden Prüfung.

**Hinweis:** Mit mehreren Features lässt sich die gesamte Modellfunktion nicht als eine einzelne 2D-Kurve darstellen. Das Notebook zeigt die Kurve für ein gewähltes Feature, während alle anderen Features auf typischen Werten konstant gehalten werden.

## 1. Datensatz hochladen

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)

# Häufige automatisch erzeugte Indexspalte entfernen
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

print(f'Datensatz: {filename}')
print(f'Zeilen: {df.shape[0]} | Spalten: {df.shape[1]}')
display(df.head())

## 2. Numerische Variablen anzeigen

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns.tolist()

print('Numerische Variablen:')
for col in numeric_columns:
    print('-', col)

## 3. Eingabevariablen und Zielvariable auswählen

Tragen Sie **eine oder mehrere numerische Spalten** in `X_VARIABLES` ein. Die Namen müssen exakt mit den Spalten im Datensatz übereinstimmen. Legen Sie außerdem das Feature für die 2D-Grafik fest.

In [ ]:
# KONFIGURATION – hier anpassen
# Beispiel Fußballer-Datensatz: Namen an Ihre tatsächlichen Spalten anpassen.
X_VARIABLES = ['goalsTotal', 'age']
Y_VARIABLE = 'marketvalue'
PLOT_FEATURE = 'goalsTotal'  # wird auf der x-Achse variiert

assert len(X_VARIABLES) > 0, 'Mindestens ein Feature auswählen.'
assert len(X_VARIABLES) == len(set(X_VARIABLES)), 'Features dürfen nicht doppelt vorkommen.'
assert Y_VARIABLE not in X_VARIABLES, 'Die Zielvariable darf kein Eingabe-Feature sein.'
assert PLOT_FEATURE in X_VARIABLES, 'PLOT_FEATURE muss in X_VARIABLES stehen.'
missing = [c for c in X_VARIABLES + [Y_VARIABLE] if c not in df.columns]
assert not missing, f'Diese Spalten fehlen im Datensatz: {missing}'
non_numeric = [c for c in X_VARIABLES + [Y_VARIABLE] if not pd.api.types.is_numeric_dtype(df[c])]
assert not non_numeric, f'Diese Spalten sind nicht numerisch: {non_numeric}'

model_df = df[X_VARIABLES + [Y_VARIABLE]].replace([float('inf'), -float('inf')], float('nan')).dropna().copy()
assert len(model_df) >= 20, 'Zu wenige vollständige Beobachtungen (mindestens 20 empfohlen).'
print('Eingabevariablen:', X_VARIABLES)
print('Zielvariable:', Y_VARIABLE)
print('Grafik-Feature:', PLOT_FEATURE)
print('Vollständige Beobachtungen:', len(model_df))

## 4. Explorative Scatterplots für alle Eingabevariablen

Jeder Plot zeigt einen **bivariaten** Zusammenhang; das spätere Modell berücksichtigt die ausgewählten Features **gemeinsam**.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(X_VARIABLES), figsize=(5 * len(X_VARIABLES), 4), squeeze=False)
for ax, feature in zip(axes[0], X_VARIABLES):
    ax.scatter(model_df[feature], model_df[Y_VARIABLE], alpha=0.45)
    ax.set(xlabel=feature, ylabel=Y_VARIABLE, title=f'{Y_VARIABLE} vs. {feature}')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Trainings-, Validierungs- und Testdaten erzeugen

Das Training lernt die Koeffizienten. Der **Validierungs-RMSE** dient zur Auswahl des Polynomgrades. Erst **danach** wird der Test-RMSE für das gewählte Modell einmalig berechnet.

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df[X_VARIABLES]
y = model_df[Y_VARIABLE]

# 60 % Training, 20 % Validierung, 20 % Test
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=42)
print(f'Training: {len(X_train)} | Validierung: {len(X_val)} | Test: {len(X_test)}')

## 6. Polynomgrade selbst festlegen

Grad 1 = linear, Grad 2 = quadratisch (inklusive Interaktionen), Grad 3 = kubisch usw. **Bei mehreren Features nimmt die Anzahl der Terme stark zu.** Starten Sie zum Beispiel mit `[1, 2, 3]`; hohe Grade können numerisch instabil sein und stark überfitten.

In [ ]:
# KONFIGURATION – zu vergleichende Polynomgrade
DEGREES = [1, 2, 3, 5]

from math import comb
assert DEGREES and all(isinstance(d, int) and d >= 1 for d in DEGREES)
p = len(X_VARIABLES)
for d in DEGREES:
    print(f'Grad {d}: {comb(p+d, d)-1} Terme (ohne Konstante)')
print('Hinweis: Bei vielen Termen gegenüber Trainingsbeobachtungen ist Overfitting wahrscheinlich.')

## 7. Multiple polynomiale Modelle trainieren und RMSE vergleichen

`PolynomialFeatures` erzeugt sowohl Potenzen als auch Kreuzprodukte (Interaktionen), z. B. `goalsTotal * age`. Die Transformation wird ausschließlich an Trainingsdaten angepasst.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
import numpy as np

results = []
models = {}

for degree in DEGREES:
    # Skalierung hilft bei unterschiedlichen Größenordnungen und hohen Potenzen
    pipeline = make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )
    pipeline.fit(X_train, y_train)
    train_pred = pipeline.predict(X_train)
    val_pred = pipeline.predict(X_val)
    rmse_train = np.sqrt(mean_squared_error(y_train, train_pred))
    rmse_val = np.sqrt(mean_squared_error(y_val, val_pred))
    term_count = pipeline.named_steps['polynomialfeatures'].n_output_features_
    results.append({'Polynomgrad': degree, 'Anzahl Terme': term_count,
                    'RMSE Training': rmse_train, 'RMSE Validierung': rmse_val})
    models[degree] = pipeline

results_df = pd.DataFrame(results).sort_values('RMSE Validierung')
display(results_df.round(3))

## 8. Polynomgrad anhand der Validierung auswählen

Wir wählen den Grad mit dem kleinsten **Validierungs-RMSE**, nicht mit dem kleinsten Trainingsfehler. **Testdaten bleiben bis zum Schluss unberührt.**

In [ ]:
best_row = results_df.iloc[0]
best_degree = int(best_row['Polynomgrad'])
print(f'Ausgewählter Polynomgrad: {best_degree}')
print(f"Validierungs-RMSE: {best_row['RMSE Validierung']:.3f}")

## 9. Modellverläufe für ein ausgewähltes Feature visualisieren

Bei mehreren Features kann eine 2D-Grafik nicht die gesamte Funktion darstellen. Wir variieren nur `PLOT_FEATURE`; **alle übrigen Features werden auf ihren Median der Trainingsdaten gesetzt**. So entsteht ein bedingter Modellverlauf (keine vollständige Darstellung des mehrdimensionalen Zusammenhangs).

Achtung: Kombinationen aus Featurewerten können dabei entstehen, die im Datensatz selten vorkommen.

In [ ]:
x_grid = np.linspace(X_train[PLOT_FEATURE].min(), X_train[PLOT_FEATURE].max(), 300)
reference = X_train.median(numeric_only=True)
X_grid = pd.DataFrame([reference.to_dict()] * len(x_grid), columns=X_VARIABLES)
X_grid[PLOT_FEATURE] = x_grid

plt.figure(figsize=(10, 6))
plt.scatter(X_train[PLOT_FEATURE], y_train, alpha=0.23, label='Trainingsdaten')
for degree in DEGREES:
    y_grid = models[degree].predict(X_grid)
    plt.plot(x_grid, y_grid, label=f'Grad {degree}')
plt.xlabel(PLOT_FEATURE)
plt.ylabel(Y_VARIABLE)
plt.title(f'Modellvergleich: {PLOT_FEATURE} variiert, andere Features = Trainingsmedian')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 10. Trainings- und Validierungs-RMSE visualisieren

In [ ]:
plot_results = pd.DataFrame(results).sort_values('Polynomgrad')
plt.figure(figsize=(8,5))
plt.plot(plot_results['Polynomgrad'], plot_results['RMSE Training'], marker='o', label='Training')
plt.plot(plot_results['Polynomgrad'], plot_results['RMSE Validierung'], marker='o', label='Validierung')
plt.xlabel('Polynomgrad')
plt.ylabel('RMSE')
plt.title('Modellkomplexität und Vorhersagefehler')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 11. Gewähltes Modell abschließend auf Testdaten bewerten

Nach Auswahl des Grades trainieren wir das Modell mit den gesamten Entwicklungsdaten (Training + Validierung) neu. Der Testdatensatz wurde für die Modellauswahl nicht verwendet.

In [ ]:
final_model = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=best_degree, include_bias=False),
    LinearRegression()
)
final_model.fit(X_dev, y_dev)
y_pred_test = final_model.predict(X_test)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f'Gewählter Grad: {best_degree} | Test-RMSE: {final_rmse:.3f}')

## 12. Leitfragen

1. Welcher Polynomgrad erzielt den kleinsten **Validierungs-RMSE**?
2. Welche Hinweise auf **Underfitting** finden Sie (hohe Fehler sowohl im Training als auch in der Validierung)?
3. Welche Hinweise auf **Overfitting** finden Sie (kleiner Trainingsfehler, aber deutlich höherer Validierungsfehler)?
4. Was verändert sich, wenn Sie statt `['goalsTotal']` mehrere Eingabevariablen wählen?
5. Was bedeutet die zunehmende Zahl polynomialer Terme und Interaktionen?
6. Warum ist die 2D-Grafik bei mehreren Features nur ein **bedingter Ausschnitt**?
7. Warum sollte der Testdatensatz nicht zur wiederholten Auswahl des Polynomgrades verwendet werden?